# 01 — Thu thập & Merge Dữ Liệu

## 1. Import & Config

In [1]:
import os
import numpy as np
import pandas as pd
import yfinance as yf

#  Đường dẫn
BASE_DIR = os.getcwd()
RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"RAW_DIR: {RAW_DIR}")

#  Ticker Yahoo Finance 
TICKERS = {
    "gold"   : "GC=F",       # Giá vàng tương lai (USD/oz)
    "dxy"    : "DX-Y.NYB",   # Dollar Index
    "sp500"  : "^GSPC",      # S&P 500
    "oil"    : "CL=F",       # Dầu thô WTI
    "vix"    : "^VIX",       # Chỉ số biến động / nỗi sợ
    "usd_vnd": "USDVND=X",   # Tỷ giá USD/VND
}

#  Thời gian 
START_DATE = "2014-01-01"
END_DATE   = None   # None = đến hôm nay

#  File output 
FILE_RAW_DATASET = os.path.join(RAW_DIR, "raw_dataset.csv")

RAW_DIR: d:\UITs subject\Năm 3\Học máy và thống kê\project\data\raw


## 2. Helper — chuẩn hóa cột date

In [2]:
def normalize_date(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df = df.copy()
    df[col] = pd.to_datetime(df[col])
    if df[col].dt.tz is not None:
        df[col] = df[col].dt.tz_localize(None)
    df[col] = df[col].dt.normalize()
    return df

## 3. Download market data từ Yahoo Finance

In [3]:
def download_market_data() -> pd.DataFrame:
    dfs = []
    for name, ticker in TICKERS.items():
        print(f"  Downloading {ticker} ({name})...", end=" ")
        try:
            raw = yf.Ticker(ticker).history(start=START_DATE, end=END_DATE)
            raw.reset_index(inplace=True)
            df  = raw[["Date", "Close"]].rename(columns={"Date": "date", "Close": name})
            df  = normalize_date(df)
            print(f"✓  {len(df):,} rows")
            dfs.append(df)
        except Exception as e:
            print(f"✗  {e}")

    # Gold là base → left join dần
    result = dfs[0]
    for df in dfs[1:]:
        result = result.merge(df, on="date", how="left")
    return result

print("=== Download market data ===")
df_market = download_market_data()
print(f"\nShape: {df_market.shape}")
print(f"Range: {df_market['date'].min().date()} → {df_market['date'].max().date()}")
df_market.tail(3)

=== Download market data ===

Shape: (3121, 7)
Range: 2014-01-02 → 2026-06-02


,date,gold,dxy,sp500,oil,vix,usd_vnd
3118,2026-05-29,4560.500000,98.910004,7580.060059,87.360001,15.320000,26326.0
3119,2026-06-01,4475.200195,99.199997,7599.959961,92.160004,16.049999,26255.0
3120,2026-06-02,4537.000000,99.172997,NaN,91.230003,NaN,26265.0


## 4. Download macro data từ FRED

> - **FEDFUNDS**: Lãi suất Fed Funds (tần suất tháng)
> - **CPIAUCSL**: Chỉ số giá tiêu dùng Mỹ (tần suất tháng)
>
> Tải tự động qua `pandas_datareader`. Nếu lỗi → tải thủ công từ https://fred.stlouisfed.org/

In [4]:
def download_macro_data():
    try:
        import pandas_datareader.data as web

        print("Tải FEDFUNDS...", end=" ")
        fed = web.DataReader("FEDFUNDS", "fred", start=START_DATE)
        fed.reset_index(inplace=True)
        fed.columns = ["date", "fed_funds"]
        fed = normalize_date(fed)
        print(f"✓  {len(fed):,} rows")

        print("Tải CPIAUCSL...", end=" ")
        cpi = web.DataReader("CPIAUCSL", "fred", start=START_DATE)
        cpi.reset_index(inplace=True)
        cpi.columns = ["date", "cpi"]
        cpi = normalize_date(cpi)
        print(f"✓  {len(cpi):,} rows")

        return fed, cpi

    except ImportError:
        raise ImportError("Chưa cài pandas_datareader → pip install pandas-datareader")

print("=== Download macro data ===")
fed, cpi = download_macro_data()
print(f"\nFed  : {fed['date'].min().date()} → {fed['date'].max().date()}")
print(f"CPI  : {cpi['date'].min().date()} → {cpi['date'].max().date()}")

=== Download macro data ===
Tải FEDFUNDS... ✓  149 rows
Tải CPIAUCSL... ✓  148 rows

Fed  : 2014-01-01 → 2026-05-01
CPI  : 2014-01-01 → 2026-04-01


## 5. Merge market + macro

In [5]:
# Bắt buộc sort trước merge_asof
df_market = df_market.sort_values("date").reset_index(drop=True)
fed        = fed.sort_values("date").reset_index(drop=True)
cpi        = cpi.sort_values("date").reset_index(drop=True)

df_raw = pd.merge_asof(df_market, fed, on="date", direction="backward")
df_raw = pd.merge_asof(df_raw,    cpi, on="date", direction="backward")

df_raw.to_csv(FILE_RAW_DATASET, index=False)
print(f"✅ Saved → {FILE_RAW_DATASET}")
print(f"Shape  : {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"\nNull counts:\n{df_raw.isnull().sum()}")
df_raw.tail(5)

✅ Saved → d:\UITs subject\Năm 3\Học máy và thống kê\project\data\raw\raw_dataset.csv
Shape  : (3121, 9)
Columns: ['date', 'gold', 'dxy', 'sp500', 'oil', 'vix', 'usd_vnd', 'fed_funds', 'cpi']

Null counts:
date          0
gold          0
dxy           1
sp500         3
oil           0
vix           3
usd_vnd       2
fed_funds     0
cpi          23
dtype: int64


,date,gold,dxy,sp500,oil,vix,usd_vnd,fed_funds,cpi
3116,2026-05-27,4447.500000,99.209999,7520.359863,88.680000,16.290001,26294.0,3.63,332.407
3117,2026-05-28,4499.299805,99.019997,7563.629883,88.900002,15.740000,26273.0,3.63,332.407
3118,2026-05-29,4560.500000,98.910004,7580.060059,87.360001,15.320000,26326.0,3.63,332.407
3119,2026-06-01,4475.200195,99.199997,7599.959961,92.160004,16.049999,26255.0,3.63,332.407
3120,2026-06-02,4537.000000,99.172997,NaN,91.230003,NaN,26265.0,3.63,332.407
